Ch 8-17 Recursion, DP, Hard, and Medium Problems --
[Open in Colab](https://colab.research.google.com/github/henry4j/-/blob/main/episode_8-B.ipynb)
* [Ch 1-3: Array, Strings, Linked Lists, Stacks, and Queues](https://colab.research.google.com/github/henry4j/-/blob/main/episode_1-3.ipynb)
* [Ch 4-6: Tree, Graph, Algerbra](https://colab.research.google.com/github/henry4j/-/blob/main/episode_4-7.ipynb)

In [ ]:
# @title import before you begin
from collections import namedtuple, Counter, defaultdict, deque
from functools import lru_cache
from heapq import heappush, heappop
from itertools import accumulate, islice, chain, count, tee
from typing import Optional, Union
import bisect

def renumerate(it, stop=None):
  return (L := list(it)).reverse() or zip(count(stop or len(L), -1), L)

def recap(kv, k, v):
  return (kv[0] + k, kv[1] + v)

def minima(iterable, key=None):
  iterator = iter(iterable)
  minima = [next(iterator, None)]
  key = key or (lambda e: e)
  min_key = minima and key(minima[0])
  for e in iterator:
    if (k := key(e)) < min_key:
      minima = [e]
      min_key = k
    elif k == min_key:
      minima.append(e)
  return minima

def maxima(iterable, key=None):
  iterator = iter(iterable)
  maxima = [next(iterator, None)]
  key = key or (lambda e: e)
  max_key = maxima and key(maxima[0])
  for e in iterator:
    if (k := key(e)) > max_key:
      maxima = [e]
      max_key = k
    elif k == max_key:
      maxima.append(e)
  return maxima

assert ["12"] == maxima(["3", "2", "12", "1"], key=len)

In [ ]:
# @title ##### def three_sum_closest(L, q=0):
def three_sum(L, q=0):
  L, n = sorted(L), len(L)
  for i in range(n-2):
    j, k = i+1, n-1
    while j < k:
      diff = sum((L[i], L[j], L[k])) - q
      if diff <0:
        j += 1
      elif diff >0:
        k -= 1
      else:
        yield (L[i], L[j], L[k])
        j, k = j+1, k-1

def three_sum_closest(L, q=0):
  L, n = sorted(L), len(L)
  minima = None
  for i in range(n-2):
    j, k = i+1, n-1
    while j < k:
      diff = sum((L[i], L[j], L[k])) - q
      if not minima or abs(diff) < min_diff:
        minima, min_diff = [(L[i], L[j], L[k])], abs(diff)
      elif min_diff == diff:
        minima.add((L[i], L[j], L[k]))
      if diff <0:
        j += 1
      elif diff >0:
        k -= 1
      else:
        j, k = j+1, k-1
  return minima

assert {(-1, -1, 2), (-1, 0, 1)} == set(three_sum([-1, 0, 1, 2, -1, -4], 0))
assert [(-4, 1, 2)] == three_sum_closest([-1, 2, 1, -4])

In [ ]:
# Given a list of intervals [start, end], collapse intervals that overlap.
def collapsed(L):  # intervals in L.
  it = iter(sorted(L, key=lambda e: e[0]))
  outcome = [next(it, None)]
  for intvl in it:
    if intvl[0] <= outcome[-1][1]:
      outcome[-1][1] = max(outcome[-1][1], intvl[1])
    else:
      outcome.append(intvl)
  return outcome

assert [[0, 3], [5, 13]] == collapsed([[0, 2], [1, 3], [5, 13], [9, 11]])
assert [[0, 13]] == collapsed([[0, 9], [1, 5], [3, 7], [9, 13]])

In [ ]:
# @title ##### # Write a function that matches UNIX filenames.
def match1(t, p, any_char):  # any_char: "?" (fnmatch), or "." (rematch)
  return p[0] == any_char or p[0] == t[0]

def fnmatch(t, p):  # P: pattern with "?", and "*".
  if p == "":  # base cases
    return t == ""
  # recursive cases:
  if p[0] == "*":
    return fnmatch(t, p[1:]) or t and fnmatch(t[1:], p)
  elif t and match1(t, p, "?"):
    return fnmatch(t[1:], p[1:])

def rematch(t, p):  # P: pattern with "." and "*".
  if p == "":  # base case:
    return t == ""
  # recursive cases:
  if len(p) >1 and p[1] == "*":
    return rematch(t, p[2:]) or t and match1(t, p, ".") and rematch(t[1:], p)
  elif t and match1(t, p, "."):
    return rematch(t[1:], p[1:])

xyz = "xyzz"
assert fnmatch(xyz, "*")
assert fnmatch(xyz, "**")
assert fnmatch("", "") and not fnmatch("x", "") and not fnmatch("", "x")
assert fnmatch(xyz, "*z") and fnmatch(xyz, "x*z") and fnmatch(xyz, "x*")
assert rematch("abc", ".b.") and rematch("abc", "a.c")
assert not rematch("aaa", "*") and rematch("aaa", "a*")
assert rematch("aaa", "aaa*") and rematch("abc", "a*.*c*")

In [ ]:
# @title Find the longest palindromic subsequence (not necessarily contiguous, nor subarray or subrange)? e.g., "ABZBA" given "XAYBZBA".
def lps(L):  # longest palindromic subsequence http://goo.gl/nTiQvX
  @lru_cache(maxsize=None)
  def prog(start, stop):  # begin(b), end(e)
    if stop - start <3:
      if L[start] == L[stop-1]:
        return [L[start:stop]]
      else:
        return [L[start:start+1], L[start+1:stop]]
    elif L[start] == L[stop-1]:
      return [L[start] + subseq + L[stop-1] for subseq in prog(start+1, stop-1)]
    else:
      return maxima(chain(prog(start+1, stop), prog(start, stop-1)), key=len)

  return prog(0, len(L))

assert ['abzba'] == list(lps('xaybzba'))
assert ['bab', 'aba'] == list(lps('abab'))

In [ ]:
# @title
# Given a string, find the minimum number of partitions where each substring is a palindrome.

In [ ]:
# @title
# Find the LCS, longest common substring https://wikipedia.org/wiki/longest_common_substring
def lcsubstring(L, R):
  @lru_cache(maxsize=None)
  def lcsuff(m, n):
    if m >0 and n >0 and L[m-1] == R[n-1]:
      start, stop = lcsuff(m-1, n-1)
      return (start, stop+1) if stop > start else (m-1, m)
    return (0, 0)

  cases = (lcsuff(m, n) for m in range(len(L)+1) for n in range(len(R)+1))
  return maxima(cases, key=lambda e: e[1] - e[0])

L, R = 'abab', 'baba'
assert ['aba', 'bab'] == [L[slice(*_)] for _ in lcsubstring(L, R)]

In [ ]:
# @title ##### def lcsubseq(L, R):
# Find the LCS, longest common subsequence, https://wikipedia.org/wiki/longest_common_subsequence
def lcsubseq(L, R):
  def prog(m, n):
    if m == 0 or n == 0:
      return {""}
    elif L[m-1] == R[n-1]:
      return {subseq + L[m-1] for subseq in prog(m-1, n-1)}
    else:
      cases = prog(m-1, n) | prog(m, n-1)
      return set(maxima(cases, key=len))

  return prog(len(L), len(R))

assert {"abace"} == lcsubseq("apbcadcqer", "rasbtaucve")

In [ ]:
# @title
# Find the optimal buying and selling strategy for k stock trades over n days.
def maximize_profit_of_stock_trades(L, k):  # at most k trades.
  def prog(n, k):  # n day stock prices, at most k trades.
    recap = lambda kv, k, v: (kv[0] + k, kv[1] + v)
    cases = (
        recap(prog(b, k-1), gain, ((b, s),))
        for b in range(0, n-1)
        for s in range(b+1, n)
        if k > 0 and (gain := L[s] - L[b]) > 0)
    return max(cases, default=(0, ()), key=lambda e: e[0]) if cases else (0, ())

  return prog(len(L), k)

L = [10, 5, 15, 20, 12, 18, 25, 10, 15]
assert (33, ((1, 3), (4, 6), (7, 8))) == maximize_profit_of_stock_trades(L, 3)

In [ ]:
# @title
# Divide a shelf of books into at most k contiguous partitions, ensuring the maximum total pages in any single partition is minimized.
def bookshelf(books, k):  # at most k partitions.
  book_acc = list(accumulate(books, initial=0))
  book_sum = lambda start, stop: book_acc[stop] - book_acc[start]

  def prog(m, k):  # m books to process, at most k partitions.
    if m > 0 and k > 0:
      cases = (prog(n, k-1) + ((n, m),) for n in range(m))
      return min(
          cases, key=lambda e: max(e, key=lambda e: book_sum(e[0], e[1])))
    else:
      return ()

  return prog(len(books), k)

books = [1, 2, 3, 4, 5, 6, 7, 8, 9]
assert ((0, 6), (6, 8), (8, 9)) == bookshelf(books, 3)

In [ ]:
# @title
def recap(kv, k, v):
  return (kv[0] + k, kv[1] + v)

def knapsack01(skus, capacity):  # a seq of costs & values.
  @lru_cache(maxsize=None)  # by capacity c and n items.
  def prog(n, c):  # returns a tuple of (a yield, items).
    if n == 0:
      return (0, ())  # a tuple of ($0, none items).
    sku = skus[n-1]
    cases = (recap(prog(n-1, c - cost), value, seq)
             for (cost, value), seq in ((sku, (n-1,)), ((0, 0), ()))
             if c >= cost)  # at most two cases
    return max(cases, key=lambda kv: kv[0])

  return prog(len(skus), capacity)

def knapsack_unbounded(skus, capacity):
  @lru_cache(maxsize=None)
  def prog(c):
    cases = (recap(prog(c - sku[0]), sku[1], (i,))
             for i, sku in enumerate(skus)
             if c >= sku[0])
    return max(cases, key=lambda kv: kv[0], default=(0, ()))

  return prog(capacity)

skus = [(2, 2), (1, 1), (4, 10), (1, 2), (12, 4)]
assert (15, (0, 1, 2, 3)) == knapsack01(skus, 16)
assert (36, (3, 3, 3, 2, 2, 2)) == knapsack_unbounded(skus, 15)

In [ ]:
class KDTree:
  def __init__(self, points, dimension=2, depth=0):
    if points:
      if depth == 0:
        points = points.copy()
      mid = len(points) // 2
      self.axis = depth % dimension
      points = sorted(points, key=lambda e: e.elements[self.axis])
      self.point = points[mid]
      self.left = KDTree(points[0:mid], dimension, depth +
                         1) if points[0:mid] else None
      self.right = KDTree(points[mid+1:], dimension, depth +
                          1) if points[mid+1:] else None

  def nearest_k(self, query, k, mins=None):
    if mins is None:
      mins = []
    heappush(mins, (-self.point.distance_sq(query), self.point))
    while len(mins) > k:
      heappop(mins)

    # r is the radius; the distance from the center (query point) of a circle.
    r = self.point.elements[self.axis] - query.elements[self.axis]
    nearer, further = (self.left, self.right) if r >0 else (self.right, self.left)
    (nearer and nearer.nearest_k(query, k, mins))
    (further and (len(mins) < k or r**2 < -mins[0][0])
     and further.nearest_k(query, k, mins))
    return mins

class KDPoint:
  def __init__(self, elements, data=None):
    self.elements, self.data = elements, data
    self.x, self.y, *_ = elements
    self.z = elements[2] if elements[2:] else None

  def distance_sq(self, other):
    return sum((other.elements[i] - e)**2 for i, e in enumerate(self.elements))

  def __repr__(self):
    return "KDPoint({0}, {1})".format(self.elements, self.data)

points = ((0.0, 0.0), (-10.1, 10.1), (12.2, -12.2), (38.3, 38.3), (179.99, 79.99))
points = [KDPoint(e, i) for i, e in enumerate(points)]
points = KDTree(points)
point0 = KDPoint((1, 2))
points.nearest_k(point0, 2)

8.1 Given a staircase with n steps, write a program to count the number of possible ways to climb it, when one can hop either 1, 2, or 3 steps at a time.

In [ ]:
# @title ##### def climb(n):
@lru_cache(maxsize=None)
def climb(n):
  if n in (1, 2):
    return n
  if n == 3:
    return 4
  return climb(n-1) + climb(n-2) + climb(n-3)

assert 13 == climb(5)
assert 24 == climb(6)

8.2 Given NxM grid, write a program to route a robot from (0, 0) to (N, M). How many possible ways are there, when the robot can move in two directions: right, and down. What if there are some spots of off-limits?

In [ ]:
# @title
# yapf:disable
def backtrack(candidate):
  def within_bounds_n_limits(r, c):  # bounds: 0s. Q: should you avoid cycles?
    return 0 <= r < m and 0 <= c and c < n and maze[r][c] == 1
  def reduce_off(candidate):
    return candidate[-1] == (m - 1, n - 1)
  def expand_out(candidate):
    r0, c0 = candidate[-1]
    for r, c in ((r0 + 1, c0), (r0, c0 + 1)):
      if within_bounds_n_limits(r, c):
        yield (r, c)

  if reduce_off(candidate):
    yield tuple(candidate)
  else:
    for e in expand_out(candidate):
      candidate.append(e)
      yield from backtrack(candidate)
      candidate.pop()

maze = [  # maze: m x n grid.
  [1, 1, 1, 1, 1, 0],
  [1, 0, 0, 0, 1, 0],
  [1, 1, 1, 1, 1, 0],
  [1, 0, 0, 0, 1, 1],
  [1, 1, 0, 0, 1, 0],
  [1, 1, 0, 1, 1, 1],
]  # yapf:disable
m, n = len(maze), len(maze[0])

expected = (
  ((0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4), (5, 4), (5, 5)),
  ((0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 4), (2, 4), (3, 4), (4, 4), (5, 4), (5, 5)),
)  # yapf: disable
bt = backtrack([(0, 0)])
assert expected == tuple(islice(backtrack([(0, 0)]), 0, 2))

8.3 Given an array of sorted integers, write a method to find a magic index where A[i] = i. What if integers are not distinct?

In [ ]:
# @title
def magic(L, lo=0, hi=None):
  if hi is None:
    hi = len(L)
  if hi - lo > 0:
    mid = (lo + hi) // 2
    if mid == L[mid]:
      return mid
    return (magic(L, lo, min(mid, L[mid] + 1))
            or magic(L, max(mid + 1, L[mid]), hi))

deq = [
    [1, 3, 3, 5, 5, 5, 7, 9, 9],  #
    [1, 3, 3, 5, 6, 6, 7, 7, 9],
    [1, 1, 3, 4, 6, 6, 7, 9, 9]
]
assert [5, 7, 1] == [magic(q) for q in deq]

In [ ]:
# @title
def bisect_left(L, x, lo=0, hi=None):
  if hi is None:
    hi = len(L)
  while lo < hi:
    mid = (lo + hi) // 2
    if x > L[mid]:
      lo = mid + 1
    else:
      hi = mid
  return lo

def bisect_right(L, x, lo=0, hi=None):
  if hi is None:
    hi = len(L)
  while lo < hi:
    mid = (lo + hi) // 2
    if x >= L[mid]:
      lo = mid + 1
    else:
      hi = mid
  return lo

def bisect_range(L, x, lo=0, hi=None):
  if hi is None:
    hi = len(L)
  args = [L, x, lo, hi]
  return (bisect_left(*args), bisect_right(*args))

L = [1, 1, 5, 5, 5, 5, 5, 7, 7]
assert [(2, 7), (0, 2), (7, 9)] == [bisect_range(L, x) for x in (5, 1, 7)]
assert [(0, 0), (9, 9)] == [bisect_range(L, x) for x in (0, 8)]

In [ ]:
# Find the exact point of transition within an array that
# contains a run of 0's and an unbounded run of 1's.
def one_sided_bsearch(L):  # algorithm design manual 4.9.2
  if L[1] == 1:
    return 1
  for n in range(1, 32):
    if L[2**n] == 1:
      import bisect
      return bisect.bisect_left(L, 1, lo=2**(n - 1) + 1, hi=2**n + 1)

assert [1, 2, 3, 4] == [
    one_sided_bsearch(a)
    for a in ((0, 1), (0, 0, 1), (0, 0, 0, 1, 1), (0, 0, 0, 0, 1))
]

8.4 Write a function to generate all subsets of a set.

In [ ]:
def all_subsets(L, stop=None):
  if (stop := len(L) if stop is None else stop) == 0:
    yield ()
  else:
    for sset in all_subsets(L, stop-1):
      yield from ((sset + (L[stop-1],)), sset)

expected = ["abc", "ab", "ac", "a", "bc", "b", "c", ""]
assert expected == list(map("".join, all_subsets("abc")))

In [ ]:
def combinations(iterable, k):  # k combinations, C(n, k), often read as "n choose k".
  pool = tuple(iterable)
  n = len(pool)
  if k > n:
    return
  indices = list(range(k))
  yield tuple(pool[i] for i in indices)
  while True:
    for i in reversed(range(k)):  # Q: any not maxed out.
      if indices[i] != i + n - k:
        break
    else:
      return  # Looks all maxed out.
    indices[i:k] = range(indices[i]+1, indices[i] + k - i+1)
    yield tuple(pool[i] for i in indices)

assert "AB AC AD BC BD CD".split(" ") == list(map("".join, combinations("ABCD", 2)))

8.7 and 8.8 Write a program to generate all permutations of a string of unique and non-unique characters.

In [ ]:
def permutate(L, i=0):  # permutate L at index i.
  if i == len(L):
    yield L
  else:
    for _, j in {L[j]: j for j in range(i, len(L))}.items():
      L[i], L[j] = L[j], L[i]
      yield from permutate(L, i+1)
      L[i], L[j] = L[j], L[i]

assert ["aab", "aba", "baa"] == list("".join(e) for e in permutate(list("aab")))

In [ ]:
def permutate_next(L):
  n = len(L)
  i = next((i for i in range(n-2, -1, -1) if L[i] < L[i+1]), -1)
  if i >=0:
    j = next(j for j in range(n-1, i, -1) if L[i] < L[j])
    L[i], L[j] = L[j], L[i]
    L[i+1:] = L[-1:i:-1]
    return L

assert [2, 1, 3, 4] == permutate_next([1, 4, 3, 2])
assert not permutate_next([4, 3, 2, 1])

In [ ]:
def permutations(iterable, r=None):
  pool = tuple(iterable)
  n = len(pool)
  r = n if r is None else r
  if r > n:
    return

  indices = list(range(n))
  cycles = list(range(n, n - r, -1))
  yield tuple(pool[i] for i in indices[:r])

  while n:
    for i in reversed(range(r)):
      cycles[i] -= 1
      if cycles[i] == 0:
        indices[i:] = indices[i+1:] + indices[i:i+1]
        cycles[i] = n - i
      else:
        j = cycles[i]
        indices[i], indices[-j] = indices[-j], indices[i]
        yield tuple(pool[i] for i in indices[:r])
        break
    else:
      return

assert ("AB AC AD BA BC BD CA CB CD DA DB DC".split(" ")  #
        == list(map("".join, permutations("ABCD", 2))))

8.9 Write a program to generate all possible, valid combinations of n-pairs of parenthesis, e.g., INPUT: 3, OUTPUT: ((())), (()()), (())(), ()(()), ()()().

In [ ]:
# @title
def combine(o, c):  # open and close counts.
  if c == 0:
    yield ""
  elif o == 0:
    yield ")" * c
  elif c == o:
    yield from ("(" + e for e in combine(o - 1, c))
  else:
    yield from ("(" + e for e in combine(o - 1, c))
    yield from (")" + e for e in combine(o, c - 1))

assert {'((()))', '(()())', '(())()', '()(())', '()()()'} == set(combine(3, 3))

In [ ]:
# @title
def interleave(L, R):
  if L == "" and R == "":
    yield ""
  elif L == "":
    yield R
  elif R == "":
    yield L
  else:
    yield from (L[0] + e for e in interleave(L[1:], R))
    yield from (R[0] + e for e in interleave(L, R[1:]))

interleaved = "ab12", "a1b2", "a12b", "1ab2", "1a2b", "12ab"
assert interleaved == tuple(interleave("ab", "12"))
assert 20 == len(tuple(interleave("abc", "123")))

8.10 Write a flood-fill method to fill in a new color until the color changes from the original color; given a point and a new color.


In [ ]:
# @title
def flood_fill_slow(graph, r, c, new):
  g = graph
  if g[r][c] != new:
    old, g[r][c] = g[r][c], new
    if r > 0 and g[r - 1][c] == old:
      flood_fill_slow(g, r - 1, c, new)
    if c > 0 and g[r][c - 1] == old:
      flood_fill_slow(g, r, c - 1, new)
    if r < len(g) - 1 and graph[r + 1][c] == old:
      flood_fill_slow(g, r + 1, c, new)
    if c < len(g[0]) - 1 and graph[r][c + 1] == old:
      flood_fill_slow(g, r, c + 1, new)
  return g

def flood_fill(graph, r, c, new):
  from collections import deque
  m, n, g = len(graph), len(graph[0]), graph  # m rows x n columns.
  old = g[r][c]
  q = deque([(r, c)])
  while q:
    r, c = q.popleft()
    if g[r][c] == old:
      w = next((w for w in reversed(range(c)) if g[r][w] != old), -1) + 1
      e = next((e for e in range(c + 1, n) if g[r][e] != old), n) - 1
      for c in range(w, e + 1):
        g[r][c] = new
        if r > 0 and g[r - 1][c] == old:
          q.append((r - 1, c))
        if r < m - 1 and g[r + 1][c] == old:
          q.append((r + 1, c))
  return g

graph = lambda: [[1, 1, 1, 1], [1, 0, 0, 1], [0, 1, 0, 1], [1, 1, 1, 1]]

# yapf:disable
assert [[1, 1, 1, 1], [1, 2, 2, 1], [0, 1, 2, 1], [1, 1, 1, 1]] == flood_fill(graph(), 1, 1, 2)
assert [[1, 1, 1, 1], [1, 3, 3, 1], [0, 1, 3, 1], [1, 1, 1, 1]] == flood_fill(graph(), 2, 2, 3)
assert [[4, 4, 4, 4], [4, 0, 0, 4], [0, 4, 0, 4], [4, 4, 4, 4]] == flood_fill(graph(), 2, 1, 4)
assert [[1, 1, 1, 1], [1, 3, 3, 1], [0, 1, 3, 1], [1, 1, 1, 1]] == flood_fill_slow(graph(), 2, 2, 3)
assert [[4, 4, 4, 4], [4, 0, 0, 4], [0, 4, 0, 4], [4, 4, 4, 4]] == flood_fill_slow(graph(), 2, 1, 4)

8.11 Given infinite # of coins (25, 10, 5, and 1 cents), write a method to count the number of ways to represent n cents.

In [ ]:
# @title
def make_amount(k, denominations, i=0):  # Processes a denom. at index i.
  d_i = denominations[i]
  return sum(
      make_amount(k - m*d_i, denominations, i+1)
      for m in range(k//d_i + 1)) if d_i > 1 else 1

assert 1 == make_amount(4, [25, 10, 5, 1])
assert 2 == make_amount(5, [25, 10, 5, 1])
assert 4 == make_amount(10, [25, 10, 5, 1])  # 4 ways to make 10 cents
assert 9 == make_amount(20, [25, 10, 5, 1])  # 13 ways to make 25 cents.

In [ ]:
# @title
def make_changes(k, denoms):  # make the change k with least coins.
  def prog(k):
    cases = ((d,) + prog(k - d) for d in denoms if k >= d)
    return min(cases, key=lambda v: len(v)) if k > 0 else ()

  return prog(k)

assert (5, 5) == make_changes(10, [7, 5, 1])
assert (7, 5, 1) == make_changes(13, [7, 5, 1])
assert (7, 7) == make_changes(14, [7, 5, 1])

8.12 Given an NxN chessboard, write a program to place eight queens so that none of them share the same row, column, or diagonal -- Visualize

In [ ]:
# @title
def peaceful_at(queens, r, c):  # peaceful at row and column (r, c)?
  return all([
      c0 != c and r - r0 != abs(c - c0)  # r > r0; it processes rows [0, n).
      for r0, c0 in enumerate(queens)
  ])

def queens_in_peace(n, queens):
  r = len(queens)
  if r == n:
    yield tuple(queens)
  else:
    for c in range(n):
      if peaceful_at(queens, r, c):
        queens.append(c)
        yield from queens_in_peace(n, queens)
        queens.pop()

assert ((1, 3, 0, 2), (2, 0, 3, 1)) == tuple(queens_in_peace(4, []))
expected = ((0,),), (), ()
assert expected == tuple(tuple(queens_in_peace(n, [])) for n in range(1, 4))

8.13 Given n boxes that cannot be rotated, but can only be stacked up, write a program to find the tallest possible stack, where the height of a stack is the sum of height of each box. Each box in the stack must be larger than the above it.



In [ ]:
# @title
def stack(boxes):

  @lru_cache(maxsize=None)
  def prog(n, L_limit=None):
    recap = lambda kv, k, v: (kv[0] + k, kv[1] + v)
    if n == 0:
      return 0, ()
    else:
      box = boxes[n - 1]
      cases = (
          prog(n - 1, L_limit),
          recap(prog(n - 1, box[0]), box[2], (n - 1,))  #
          if L_limit is None or box[0] <= L_limit else (0, ()))
      return max(cases, key=lambda kv: kv[0])

  height, choices = prog(len(boxes))
  return height, choices

# yapf:disable
boxes = [  # sorted by W (width).
  (100, 30, 10), (200, 40, 10), (300, 50, 21),
  (10, 200, 30), (20, 300, 10), (30, 400, 20)
]
assert (60, (3, 4, 5)) == stack(boxes)

8.14 Given a boolean equation, write a program to count the number of ways to parenthesize the expression such that equation is true, e.g., INPUT: Expression: 1^0|0|1, Desired Result: false(0), OUTPUT: 2 ways: 1^((0|0)|1) and 1^(0|(0|1)).

In [ ]:
# @title
def express(expression, bit):
  n = len(expression) // 2
  if n < 2:
    return 1 if bit == eval(expression) else 0
  else:
    m = 0
    for p in range(n):
      opr, opd1, opd2 = (expression[2 * p + 1], expression[0:2 * p + 1],
                         expression[2 * p + 2:])
      opd1_1, opd1_0 = (express(a, b) for a, b in ((opd1, 1), (opd1, 0)))
      opd2_1, opd2_0 = (express(a, b) for a, b in ((opd2, 1), (opd2, 0)))
      if opr == '|':
        m += ((opd1_1 * opd2_1 + opd1_1 * opd2_0 +
               opd1_0 * opd2_1) if bit == 1 else (opd1_0 * opd2_0))
      if opr == '&':
        m += ((opd1_1 * opd2_1) if bit == 1 else
              (opd1_1 * opd2_0 + opd1_0 * opd2_1 + opd1_0 * opd2_0))
      if opr == '^':
        m += ((opd1_1 * opd2_0 + opd1_0 * opd2_1) if bit == 1 else
              (opd1_1 * opd2_1 + opd1_0 * opd2_0))
    return m

assert 2 == express("1^0|0|1", 0)

In [ ]:
# @title


In [ ]:
# @title


10.3 Given an array of sorted integers that has been rotated a number of times, write code to find an element in the array.

In [ ]:
# @title
from collections import deque

def index_in_cycle(L, key, start=0, stop=None):
  if stop is None:
    stop = len(L)
  while stop - start >0:
    mid = (start+stop) // 2
    if L[mid] == key:
      return mid
    if L[mid] > L[stop-1]:
      if L[start] <= key < L[mid]:  # monotonicLlly rising.
        stop = mid
      else:
        start = mid+1
    else:
      if L[mid] < key <= L[stop-1]:  # monotonically rising.
        start = mid+1
      else:
        stop = mid

deq = [deque([7, 8, 0, 1, 2, 3, 4, 5, 6]), deque([3, 4, 5, 6, 7, 8, 0, 1, 2])]
for q in deq:
  for e in (0, 3, 5, 7):
    assert e == q[index_in_cycle(q, e)]

def min_in_cycle(L, start=0, stop=None):
  if stop is None:
    stop = len(L)
  if stop - start >1:
    if L[mid := (start+stop-1) // 2] > L[stop-1]:
      return min_in_cycle(L, mid+1, stop)
    else:  # RHS is monotonically rising.
      return min_in_cycle(L, start, mid+1)
  else:
    return start

assert [0, 0, 1] == [min_in_cycle(L) for L in ([6], [6, 7], [7, 6])]
assert [2, 6] == [min_in_cycle(q) for q in deq]

In [ ]:
# @title
def peak(L, start=0, stop=None):
  if stop is None:
    stop = len(L)
  mid = (start+stop-1) // 2
  if L[mid-1] < L[mid] > L[mid+1]:
    return mid
  elif L[mid-1] < L[mid] < L[mid+1]:
    return peak(L, mid+1, stop)
  else:
    return peak(L, start, mid)

assert 6 == peak([1, 5, 7, 9, 15, 18, 21, 19, 14])

10.5 Given an array of sorted strings, which is interspersed with empty strings, write a method to find the location of a given string, e.g., find 'ball' in ['at', '', 'ball', '', '', 'car', '', 'dad', '', ''].

In [ ]:
# @title
def index(L, key, start=0, stop=None):
  if stop is None:
    stop = len(L)
  if stop - start > 0:
    lo = hi = mid = (start + stop - 1) // 2
    while not L[mid]:
      if lo < start and hi >= stop:
        return
      if lo >= start and L[lo]:
        mid = lo
      elif hi < stop and L[hi]:
        mid = hi
      else:
        lo, hi = lo - 1, hi + 1
    return (index(L, key, start, mid) if key < L[mid] else
            index(L, key, mid + 1, stop) if key > L[mid] else mid)

assert 1 == index(["", "abc", "dos", "", "", "ijk", "xyz"], "abc")
assert 6 == index(["", "abc", "dos", "", "", "ijk", "xyz"], "xyz")

10.9 Given an NxM matrix in which each row and each column is sorted in ascending order, write a method to find an element.

In [ ]:
# @title
def find_in_grid(g, q, rows, cols):  # 3/4-section; not bisection.
  if rows[1] > rows[0] and cols[1] > cols[0]:
    r, c = ((e[1] + e[0]) // 2 for e in (rows, cols))
    if q < g[r][c]:  # find in 3/4 of the section.
      return (find_in_grid(g, q, (rows[0], r), cols)
              or find_in_grid(g, q, (r, rows[1]), (cols[0], c)))
    elif q > g[r][c]:  # find in 3/4 of the section.
      return (find_in_grid(g, q, (r + 1, rows[1]), cols)
              or find_in_grid(g, q, (rows[0], r + 1), (c + 1, cols[1])))
    else:
      return (r, c)

g = [
    [11, 23, 35, 47],  #
    [22, 34, 38, 58],
    [33, 39, 57, 62],
    [44, 45, 61, 69]
]
rows, cols = (0, len(g)), (0, len(g[0]))
assert [(0, 3), (3, 3), (0, 0), (3, 0), (2, 1), (3, 2)]  \
    == [find_in_grid(g, q, rows, cols) for q in (47, 69, 11, 44, 39, 61)]

In [ ]:
# @title
def find_in_rng(a1, a2, rng=None, rng1=None, rng2=None):
  rng1 = rng1 or [0, len(a1)]
  rng2 = rng2 or [0, len(a2)]
  m, n = rng1[1] - rng1[0], rng2[1] - rng2[0]
  if rng is None:
    k, z = divmod(m + n, 2)
    rng = [k, k + 1 if z else k + 2]
  if n == 1:
    rng1[0] += rng[0] - 1
    rng[0], rng[1] = 0, rng[1] - rng[0]
  elif m == 1:
    rng2[0] += rng[0] - 1
    rng[0], rng[1] = 0, rng[1] - rng[0]
  if rng[0] == 0:
    res = []
    for _ in range(rng[0], rng[1]):
      if rng1[0] < len(a1) and rng2[0] < len(a2):
        if a1[rng1[0]] < a2[rng2[0]]:
          res.append(a1[rng1[0]])
          rng1[0] += 1
        else:
          res.append(a2[rng2[0]])
          rng2[0] += 1
      elif rng1[0] < len(a1):
        res.append(a1[rng1[0]])
        rng1[0] += 1
      elif rng2[0] < len(a2):
        res.append(a2[rng2[0]])
        rng2[0] += 1
    return res
  ext1 = rng[0] * m // (m + n)
  ext2 = rng[0] - ext1
  cmp0 = a1[rng1[0] + ext1] - a2[rng2[0] + ext2]
  if cmp0 < 0:
    rng[0], rng[1] = rng[0] - ext1, rng[1] - ext1
    rng1[0] = rng1[0] + ext1
    rng2[1] = rng2[0] + ext2
  else:
    rng[0], rng[1] = rng[0] - ext2, rng[1] - ext2
    rng1[0] = rng1[0] + ext2
    rng2[1] = rng2[0] + ext1
  return find_in_rng(a1, a2, rng, rng1, rng2)

find_in_rng(list(range(10, 20)), list(range(20, 30)), [0, 12])

10.10 Design and implement a data structure and an algorithm that can track a stream of numbers, and tell the rank of a value x (the number of values less than or equal to x).

In [ ]:
# @title
class BTree:  # tree has a value (key=x, rank=# of less than equal to x).
  def __init__(self, value=None, left=None, right=None):
    self.value, self.left, self.right = value, left, right

  def track(self, x):
    if x <= self.value[0]:
      if self.left:
        self.left.track(x)
      else:
        self.left = BTree([x, 1])
      self.value[1] += 1
    else:
      if self.right:
        self.right.track(x)
      else:
        self.right = BTree([x, 1])
    return self

def rank(tree, x):
  if x < tree.value[0]:
    return rank(tree.left, x) if tree.left else 0
  elif x > tree.value[0] and tree.right:
    return tree.value[1] + rank(tree.right, x)
  else:
    return tree.value[1]

# tree:    20(4) has a value of value, and # of nodes of smaller values.
#        /    \
#     15(3)   29(2)
#     /       /
#   10(1)    23(0)
#  /   \      \
# 5(0) 13(0)  26(0)
tree = BTree([2**31, -1]).track(20).track(15).track(10).track(5)
tree = tree.track(13).track(29).track(23).track(26)
assert 5 == rank(tree, 20)
assert 8 == rank(tree, 29)
assert 6 == rank(tree, 23)
assert 6 == rank(tree, 25)

In [ ]:
# @title ##### Interval tree https://www.geeksforgeeks.org/interval-tree/
class ITree:
  def __init__(self, key, left=None, right=None, value=None):
    self.key, self.left, self.right, self.value = key, left, right, value
    is_not_none = lambda e: e is not None
    maxima = (c and c._max or None for c in (self.left, self.right))
    self._max = _max = max(filter(is_not_none, (key[1], *maxima)))

  def __repr__(self):
    m = len(values := list(v for (k, v) in vars(self).items() if k[0] != "_"))
    n = next((i for i, e in enumerate(reversed(values)) if e is not None), m)
    return f"ITree({repr(values[:m-n])[1:-1]})"

  def __setitem__(self, key, value):
    self._max = max(self._max, key[0])
    if self.key == key:
      self.value = value
    elif key[0] < self.key[0]:
      if self.left:
        self.left[key] = value
      else:
        self.left = ITree(key, value=value)
    else:
      if self.right:
        self.right[key] = value
      else:
        self.right = ITree(key, value=value)

  def __getitem__(self, key):
    if key[0] <= self.key[0] <= key[1] or key[0] <= self.key[1] <= key[1]:
      yield self
    if self.left and key[0] <= self.left._max:
      yield from self.left[key]
    if self.right and key[1] >= self.right.key[0]:
      yield from self.right[key]

tree = ITree((15, 20))
for intvl in ((10, 30), (5, 20), (12, 15), (17, 19), (30, 40)):
  tree[intvl] = None

assert [(10, 30), (5, 20)] == list(e.key for e in tree[(4, 11)])
assert [(10, 30), (30, 40)] == list(e.key for e in tree[(21, 30)])

16.1 Write a program to swap two numbers in-place without using additional variables.  
16.2 Write a program to find the frequency of occurrences of any given word in a book.  
16.3 Given two line segments, write a program to identify the point, where they cross each other.  
16.4 Write a program to test if a player has won a tic-tac-toe game.  
16.5 Write a method to compute the number of trailing zeros in n factorial.
```python
zeros = lambda n: (q := n // 5) + (zeros(q) if q > 4 else 0)  # quotient!
assert (0, 1, 4, 6, 18) == tuple(zeros(n) for n in (4, 5, 24, 25, 75))
```
16.6 Given two integer arrays, find the pair of integers (one from each array) with the smallest absolute difference. How about sorting and scanning two arrays?  
16.7 Write a program that finds the maximum of two numbers. Do not use if-else and comparisons.  

In [ ]:
# @title ##### 16.8 Spell out English words for numerical figures. Write "One Thousand Two Hundred Thirty Four" for 1234.
digits = [None, "One", "Two", "Three", "Four", "Five", "Six", "Seven", "Eight", "Nine"]
teens = [
  "Ten",
  "Eleven",
  "Twelve",
  "Thirteen",
  "Fourteen",
  "Fifteen",
  "Sixteen",
  "Seventeen",
  "Eighteen",
  "Nineteen",
]
tens = [
  None,
  "Ten",
  "Twenty",
  "Thirty",
  "Forty",
  "Fifty",
  "Sixty",
  "Seventy",
  "Eighty",
  "Ninety",
]
bigs = [None, "Thousand", "Million"]

def read(d, x1000=0):
  if d < 0:
    return ("Negative",) + read(-decimal, x1000)
  if d == 0:
    return ("Zero",) if x1000 == 0 else ()
  return (
    read(d // 1000, x1000 + 1)
    + read3d(d % 1000)
    + ((bigs[x1000],) if x1000 > 0 else ())
  )

def read3d(d):
  if d == 0:
    return ()
  if d >= 100:
    return (digits[d // 100], "Hundred") + read3d(d % 100)
  if 10 < d < 20:
    return (teens[d - 10],)
  if d == 10 or d > 19:
    return (tens[d // 10],) + read3d(d % 10)
  return (digits[d],)

# fmt:off
assert ('Zero',) == read(0)
assert ('Ten', 'Million', 'Two', 'Hundred', 'Twelve', 'Thousand', 'Twenty', 'One') == read(10212021)

16.9 Create functions to perform multiplication, subtraction, and division on integers. You can only use additions.

16.10 Given a list of people and their birth and death years, calculate the year with the maximum number of living individuals.



In [ ]:
# @title
# Create functions to perform multiplication, subtraction, and division on integers, using only addition.

In [ ]:
# @title ##### 16.11 Given two different lengths of wooden planks, short and long, and a specific number of planks K, find all possible total lengths of a diving board that can be constructed using exactly K planks. Each plank used must be either short or long.
from functools import lru_cache
from itertools import chain

def diving_boards(k: int, short: int, long_: int):
  @lru_cache(maxsize=None)
  def prog(franks: int, short_franks, long_franks):
    if franks == 0:
      yield 0
    else:
      yield from chain(
        (e + short for e in prog(franks - 1, short_franks + 1, long_franks)),
        (e + long_ for e in prog(franks - 1, short_franks, long_franks + 1)),
      )
  yield from prog(k, 0, 0)

assert [6, 7, 8, 9] == list(diving_boards(3, 2, 3))

16.12 Given an XML document and a mapping of XML tags to integer values, write a program that compresses an XML doc with encoding rules.
* Element: Tag Attributes END Children END
* Attribute: Tag Value
* END: 0
* Tag: A predefined integer mapping, e.g., family -> 1, person ->2, firstName -> 3, lastName -> 4, state
* Value: A string value


16.13 Given two squares on a 2D plane, find a line that would cut these two squares in half. Each square has top and bottom sides running parallel to the x-axis.  
16.14 Given a set of points on a 2D plane, find the line that passes through the maximum number of points.  

In [ ]:
# @title ##### 16.15 Imagine a secret code with four colors: Red (R), Yellow (Y), Green (G), and Blue (B). A game player tries to guess the secret code by picking four colors. Scores are given as a hit, if the color and the place both are right, or a pseudo-hit, if the color is right, but the place is not right.
from collections import Counter

def score(codes, guess):
  i_hits = {i for i, c, g in zip(range(4), codes, guess) if c == g}
  c_codes = Counter(c for i, c in enumerate(codes) if i not in i_hits)
  c_guess = Counter(g for i, g in enumerate(guess) if i not in i_hits)
  return len(i_hits), sum(min(h, c_codes.get(g, 0)) for g, h in c_guess.items())

score("RGBY", "GGRR")

In [ ]:
# @title ##### 16.16 Given an array of integers, find the minimum subarray that, when sorted, sorts the entire array.
def unordered(L):  # returns (start, stop).
  def renumerate(it, stop=None):
    return (L := list(it)).reverse() or zip(count(stop or len(L), -1), L)

  maxima = accumulate(L, max)  # [0, 3, 3, 3, 4, 5, 6]
  stop = next((s for s, (e, x) in renumerate(zip(L, maxima)) if e < x), 0)
  minima = reversed(list(accumulate(reversed(L), min)))
  start = next((s for s, (e, n) in enumerate(zip(L, minima)) if e > n), 0)
  return start, stop

cases = (range(6), [1, 0, 2, 3, 4], [0, 1, 2, 4, 3], [0, 1, 3, 2, 4])
assert [(0, 0), (0, 2), (3, 5), (2, 4)] == [unordered(c) for c in cases]

In [ ]:
# @title ##### 16.17 Given an array of integers (both positive and negative), write a program to find the max sum sub-array (contiguous sequence).
# Kadane's algorithm http://en.wikipedia.org/wiki/Maximum_subarray_problem
def maxsum_subarray(a):
    sum_ = max_sum = left = 0
    max_range = (0, 0)
    for right, e in enumerate(a):
        if sum_ > 0:
            sum_ = sum_ + e
        else:
            left = right; sum_ = e
        if sum_ > max_sum:
            max_sum = sum_; max_range = (left, right)
    return (max_sum, max_range)

assert (5, (1, 4)) == maxsum_subarray([-2, 1, 3, -3, 4, -2, -1, 3])

In [ ]:
# @title ##### 16.18 Pattern Matching : You are given two strings, patternand value.The patternstring consists of just the letters aand b, describing a pattern within a string. For example, the string catcatgocatgomatches the pattern aabab(where catis aand gois b). It also matches patterns like a, ab, and b. Write a method to determine if value matches pattern.
def match(text: str, pattern: str):
  def prog(T, start, stop, **kwargs):
    if stop == start:
      if T == "":
        yield kwargs
    else:
      c = pattern[start]
      if lead := kwargs.get(c, None):
        if T.startswith(lead):
          yield from prog(T[len(lead) :], start + 1, stop, **kwargs)
      else:
        m = len(T)
        for k in range((m + 1) // pattern[start:stop].count(c), 1, -1):
          lead, tail = T[:k], T[k:]
          yield from prog(tail, start + 1, stop, **(kwargs | {c: lead}))
  yield from prog(text, 0, len(pattern))

# list(match("catcatgocatgo", "a"))
# list(match("catcatgocatgo", "ab"))
# list(match("catcatgocatgo", "aba"))
# list(match("catcatgocatgo", "aabab"))

16.19 Pond Sizes: You have an integer matrix representing a plot of land, where the value at that location
represents the height above sea level. A value of zero indicates water. A pond is a region of water
connected vertically, horizontally, or diagonally. The size of the pond is the total number of
connected water cells. Write a method to compute the sizes of all ponds in the matrix.
EXAMPLE
Input:
0 2 1 0
0 1 0 1
1 1 0 1
0 1 0 1
Output: 2, 4, 1 (in any order)

In [ ]:
# @title ##### 16.21 Given two integer arrays, find two integers (one from each array) that can be swapped to equalize their sums.
lhs, rhs = [1, 2, 3, 5], [4, 5, 6]
q, r = divmod(sum(lhs) - sum(rhs), 2)
assert r == 0
matcher, seen = {q+e for e in rhs}, set()
for e in lhs:
  if e in matcher and e not in seen:
    print((e, e-q))
    seen |= {e}

In [ ]:
# @title ##### 16.22 An ant starts on an infinite grid of alternating white and black squares, facing right. At each step:
# On a white square, it flips the color, turns 90° clockwise, and moves forward.
# On a black square, it flips the color, turns 90° counterclockwise, and moves forward.
# Write a program to simulate the ant's movement for K steps and print the final grid configuration. The grid structure must be designed by you. The input is the integer K, and the program should print the grid without returning anything. A possible method signature is void printKMoves(int K).





In [ ]:
# @title ##### 16.23 Given a function rand5() that generates a random number between 1 and 5 (inclusive), write a function that generates a random number between 1 and 7 (inclusive).
def rand5():
  import random
  return random.randrange(5) + 1

def rand7():
  rand25 = 22
  while rand25 > 21:  # [1, 21]
    rand25 = 5 * (rand5() - 1) + rand5()  # [1, 25]
  return rand25 % 7 + 1  # [1, 7].

from collections import Counter
c = Counter(rand7() for _ in range(7000))
assert all(900 < e < 1100 for e in c.values())

In [ ]:
# @title ##### 16.24 Write a program to find all pairs from an integer array, which sum to integer x.
def pairs(L, x):
  matcher = set()
  seen = set()
  for e in L:
    if e in matcher and e not in seen:
      pair = tuple(sorted((e, x - e)))
      seen |= {*pair}
      yield pair
    else:
      matcher |= {x - e}

assert [(1, 2), (0, 3)] == list(pairs([1, 2, 3, 1, 0, -1], 3))

In [ ]:
# @title ##### 16.25 Design an LRU cache with a max size, that evicts the least recently used when full.
# You can assume the keys are integers and the values are strings.
class DNode:
  def __init__(self, value, pred=None, succ=None):
    self.value, self.pred, self.succ = value, pred, succ

class LRUCache:
  def __init__(self, capacity=1):
    self.capacity = capacity
    self.ha5h = {}
    self.head = self.tail = None
  def put(self, k, v):
    k in self.ha5h and self.pop_node(self.ha5h[k])
    self.append_node(DNode((k, v)))
    self.ha5h[k] = self.tail
    while len(self.ha5h) > self.capacity:
      del self.ha5h[self.pop_node(self.head).value[0]]
    return self
  def get(self, k):
    if k in self.ha5h:
      self.pop_node(self.ha5h[k])
      self.append_node(self.ha5h[k])
      return self.tail.value[1]
  def append_node(self, node):  # push at tail
    node.succ, node.pred = None, self.tail
    if self.tail:
      self.tail.succ = node
      self.tail = node
    else:
      self.head = self.tail = node
  def pop_node(self, node):
    if self.head != node:
      node.pred.succ = node.succ
    else:
      self.head = self.head.succ
      self.head.pred = None
    if self.tail != node:
      node.succ.pred = node.pred
    else:
      self.tail = self.tail.pred
      self.tail.succ = None
    return node

c = LRUCache(3).put(1, "a").put(2, "b").put(3, "c")
assert "a" == c.get(1) and "b" == c.get(2)
assert c.put(4, "d").get(3) is None
assert c.put(5, "e").get(1) is None
assert "b" == c.put(5, "e").get(2)

In [ ]:
# @title
class CircularBuffer:  # http://www.programiz.com/python-programming/exceptions
  def __init__(self, capacity):
    self._a = [None] * (capacity+1)
    self._head = self._tail = 0

  def enq(self, v):
    if self.full():
      raise RuntimeError("This buffer is full.")
    self._a[self._tail] = v
    self._tail = (self._tail+1) % len(self._a)
    return self

  def deq(self):
    if self.empty():
      raise RuntimeError("This buffer is empty.")
    v = self._a[self._head]
    self._head = (self._head+1) % len(self._a)
    return v

  def empty(self):
    return self._head == self._tail

  def full(self):
    return self._head == (self._tail+1) % len(self._a)

# yapf: disable
buffer = CircularBuffer(3)
assert buffer.empty() and not buffer.full()
assert buffer.enq(10).enq(20).enq(30).full()
try: buffer.enq(40)
except RuntimeError: pass
assert [10, 20, 30] == [buffer.deq() for _ in range(3)]
assert buffer.empty()
try:  buffer.deq()
except RuntimeError: pass

In [ ]:
# @title ##### 16.26 Calculate the result of a given arithmetic expression, which only includes positive integers and the basic operations of addition, subtraction, multiplication, and division.
ops = {"*": 1, "/": 1, "+": 2, "-": 2, "(": 3}

def postfix(a):
  Q, stack = [], []
  for e in a:
    if e == "(":
      stack.append(e)
    elif e == ")":
      while stack[-1] != "(":
        Q.append(stack.pop())
      stack.pop()
    elif e not in ops:
      Q.append(e)
    else:
      while stack and ops[stack[-1]] <= ops[e]:
        Q.append(stack.pop())
      stack.append(e)
  while stack:
    Q.append(stack.pop())
  return Q

def evaluate(Q):  # rpn Q: reverse polish notation.
  stack = []
  for e in Q:
    if e in ops:
      e = eval("{2}{1}{0}".format(stack.pop(), e, stack.pop()))
    stack.append(e)
  return stack.pop()

assert "234*+23+4*+" == "".join(postfix(list("2+3*4+(2+3)*4")))
assert 34 == evaluate(postfix(list("2+3*4+(2+3)*4")))

In [ ]:
# @title ##### 18.1 Write a functions that adds two numbers. You should not use the addition (+) arithmetic operator.
def sum_(a, b):  # a: ones, b: carries
  return sum_(a ^ b, (a & b) << 1) if b > 0 else a

assert 10 == sum_(7, 3)

In [ ]:
# @title ##### 18.2 Write a function to shuffle a deck of cards. Each of the 52! permutations of the deck has to be equally probable.
def fisher_yates_shuffle(L):
  n = len(L)
  for i in range(n - 1):
    j = i + random.randrange(n - i)  # j is in [i, n-1] from i + [0, n-i-1].
    L[j], L[i] = L[i], L[j]
  return L

In [ ]:
# @title
# 18.3 Write a function to randomly sample m integers out of n integers. Each element must have equal probability of being chosen.
from collections import Counter
import random

def reservoir_samples(iterable, k):
  samples = []
  count = 0
  for e in iterable:
    count += 1
    if len(samples) < k:
      samples.append(e)
    elif (i := random.randrange(count)) < k:
      samples[i] = e
  return samples

def weighted_choice(weights):
  rand = random.randrange(sum(weights))
  return next(i for i, e in enumerate(accumulate(weights)) if e > rand)

assert 3 == len(reservoir_samples(iter(range(100)), 3))
weights = [20, 30, 50]
c = Counter(weighted_choice(weights) for _ in range(1000))
sorted(c.items(), key=lambda k: c[k])

In [ ]:
# @title
# 17.5 Write a program to find the longest subarray that has an equal number of letters and digits.
def longest_subarray_with_alpha_nums_balanced(s):
  maxima = []
  max_key = 0
  hash = {0: 0}  # start indices for acc.
  acc = accumulate([-1, 1][c.isalpha()] for c in s)
  for stop, e in enumerate(acc, 1):
    if e not in hash:
      hash[e] = stop
    else:
      key = stop - hash[e]
      if key > max_key:
        maxima = [(hash[e], stop)]
        max_key = key
      elif key == max_key:
        maxima.append((hash[e], stop))
  return maxima

assert [(0, 4), (3, 7)] == longest_subarray_with_alpha_nums_balanced("a12b55c")

In [ ]:
# @title ##### 17.6 Write a function to count the number of 2s between O and n.
def count_2s_upto_e(b):  # 1 up to E+1, 20 up to E+2, 300 up to E+3.
  return b * 10 ** (b - 1)

def count_2s_upto(n):
  count, e, p = 0, 0, n  # preceding(p) precedes number(n).
  while p > 0:
    d = p % 10  # we look at a digit (rE+e) in each iteration.
    count += d * count_2s_upto_e(e)
    if d > 3:
      count += 10**e  # e.g., count += 100 for [200, 299], when p >299 and e=2
    if d == 2:
      count += n % 10**e + 1  # e.g., count += 56 for [200, 255], when n: x255, e: 2.
    p //= 10
    e += 1
  return count

assert 1 == count_2s_upto(9)
assert 20 == count_2s_upto(99)
assert 300 == count_2s_upto(999)
assert 3059 == count_2s_upto(6789)

In [ ]:
# @title ##### 17.7 Each year, the government releases a list of the 10,000 most common baby names and their frequencies. Some names, like "John" and "Jon," are variations of the same name but are listed separately. Given two lists—one with names and frequencies, and another with equivalent name pairs—design an algorithm to consolidate frequencies for equivalent names. If "John" is equivalent to "Jon," and "Jon" to "Johnny," all three are treated as equivalent (transitive and symmetric relationship). The final list should group equivalent names and use any as the primary name.
def consolidate_baby_names(counter, synonyms):
  id_map = {}
  for lhs, rhs in synonyms:
    lhs_id, rhs_id = id_map.setdefault(lhs, lhs), id_map.setdefault(rhs, rhs)
    if lhs_id != rhs_id:
      id_ = ":".join(sorted(lhs_id.split(":") + rhs_id.split(":")))
      for name in id_.split(":"):
        id_map[name] = id_
      lhs_count, rhs_count = counter.pop(lhs_id, 0), counter.pop(rhs_id, 0)
      counter[id_] = lhs_count + rhs_count
  return counter

counter = dict(john=15, jon=12, chris=13, kris=4, christopher=19)
synonyms = (("jon", "john"), ("chris", "kris"), ("chris", "christopher"))
consolidate_baby_names(counter, synonyms)

In [ ]:
# @title ##### 17.8 Write a program to design a circus of the largest tower of people standing atop one another's shoulders. For practical and aesthetic reasons, each person must be both shorter and lighter than the person below him or her.
def maxima(iterable, key=lambda e: e):
  maxima, max_key = None, None
  for k, e in zip(map(key, (it := tee(iterable, 2))[0]), it[1]):
    if not maxima or k > max_key:
      maxima, max_key = [e], k
    elif k == max_key:
      maxima.append(e)
  return maxima

def lis(L):
  @lru_cache(maxsize=None)
  def lisuff(i):
    cases = (
        e + (L[i],)  #
        for j in range(i)
        for e in lisuff(j)
        if e[-1] <= L[i])  # 0 <= j < i
    return maxima(cases, key=len) or [(L[i],)]

  return maxima((e for i in range(len(L)) for e in lisuff(i)), key=len)

def lisubsequence(L):
  memos = []
  for i, e in enumerate(L):
    for j, m in reversed(list(enumerate(memos))):
      if m[-1] <= L[i]:
        break
    else:
      j = -1
    m = memos[j] + [L[i]] if j != -1 else [L[i]]
    if j < len(memos)-1:
      memos[j+1] = m
    else:
      memos.append(m)
  return memos[-1]

assert [(1, 5, 6), (1, 2, 3)] == lis((7, 8, 1, 5, 6, 2, 3))
assert [1, 2, 3] == lisubsequence([7, 8, 1, 5, 6, 2, 3])

In [ ]:
# @title
# 17.9 Write a program to find the k-th number such that the only prime factors are 3, 5, and 7.
from collections import deque, namedtuple
from itertools import islice

def multiple_of(prime_factors):
  queues = {p: deque([p]) for p in prime_factors}
  while True:
    yield (x := min(q[0] for q in queues.values()))
    for prime, q in queues.items():
      if q[0] == x:
        q.popleft()
      q.append(prime * x)

assert [3, 5, 7] == list(islice(multiple_of((3, 5, 7)), 3))
assert [15, 21, 25] == list(islice(multiple_of((3, 5, 7)), 4, 7))

In [ ]:
# @title ##### 17.10 Given an array of positive integers, determine the majority, if any. A majority is an element that occurs more than half the times in the array.
def boyer_moore_majority(L):
  candidate, votes = None, 0
  for e in L:
    if votes == 0:
      candidate, votes = e, 1
    else:
      votes += 1 if e == candidate else -1
  return candidate if L.count(candidate) > len(L) // 2 else None

assert 1 == boyer_moore_majority([1, 1, 1, 1, 2, 3, 4])

![](https://i.sstatic.net/mTTN8.png)

In [ ]:
# @title ##### 17.11 Given a large text file containing words, find the shortest distance between k words in terms of the number of words.
from collections import deque
import heapq

def min_word_window(k_indices):  # O(L*logK)
  # positions, e.g., [[0, 89, 130], [95, 123, 177, 199], [70, 105, 117]]
  k_indices = [deque(e) for e in k_indices]
  min_window = window = [e.popleft() for e in k_indices]  # [0, 95, 70]
  heapq.heapify((heap := [(w, k) for k, w in enumerate(window)]))
  while k_indices[heap[0][1]]:
    _, k = heapq.heappop(heap)
    window[k] = k_indices[k].popleft()
    min_window = min(min_window, window, key=lambda w: max(w) - min(w))
    heapq.heappush(heap, (window[k], k))
  return min_window

k_word_indices = [[0, 89, 130], [95, 123, 177, 199], [70, 105, 117]]
assert [130, 123, 117] == min_word_window(k_word_indices)

In [ ]:
# @title ##### 17.12 Write a program to convert a binary search tree to a doubly linked list. Keep the values in order while converting in-place.
from itertools import islice

class BTree:
  def __init__(self, value=None, left=None, right=None, parent=None):
    self.value, self.left, self.right, self.parent = value, left, right, parent
  def __repr__(self):
    m = len(values := list(vars(self).values()))
    n = next((i for i, e in enumerate(reversed(values)) if e is not None), m)
    return f"BTree({repr(values[:m-n])[1:-1]})"
  @classmethod
  def from_values(cls, values, start=0, stop=None):
    if stop is None:
      stop = len(values)
    if stop - start > 0:
      mid = (start + stop - 1) // 2
      L = BTree.from_values(values, start, mid)
      R = BTree.from_values(values, mid + 1, stop)
      return cls(values[mid], L, R, None)

def iterate(tree):  # returns an iterator.
  stack = []
  while tree or stack:
    if tree:
      stack.append(tree)
      tree = tree.left
    else:
      yield (tree := stack.pop())
      tree = tree.right

b5 = BTree.from_values(range(10))
it = iterate(b5)
head = pred = next(it)
for tree in it:
  tree.left = pred
  pred.right = tree
  pred = tree

tree = head
while tree:
  tree = tree.right

In [ ]:
# @title ##### 17.13 Given a text with all spaces, punctuation, and capitalization removed, e.g., "iresetthecomputeri tstilldidntboot" from "I reset the computer. It still didn’t boot!". Respace the text while keeping most letters recognizable. Don't worry about capitalization or punctuation for now.
def respace(text, vocas):
  from functools import lru_cache
  vocas = dict.fromkeys(sorted(vocas, key=len, reverse=True))
  recap = lambda kv, k, v: (kv[0] + k, kv[1] + v)  # recap of (minutes, sequence).
  @lru_cache(maxsize=None)
  def prog(m):
    if m > 0:
      cases = [
        recap(prog(n), 0 if text[n:m] in vocas else m - n, (m,))
        for n in range(m - 1, -1, -1)
      ]
      return min(cases, key=lambda e: e[0])
    else:
      return (0, ())
  compose = lambda c: (text[i > 0 and c[i - 1] : stop] for i, stop in enumerate(c))
  rubbish, stops = prog(len(text))
  return rubbish, " ".join(compose(stops))

vocas = ["you", "reset", "the", "computer", "still", "didnt", "boot"]
text = "youresetthecomputeritstilldidntboot"
respace(text, vocas)

In [ ]:
# @title ##### 17.14 Write a program to find the smallest million numbers in billion numbers. Your computer can hold all one billion numbers.
import random

def quicksort_k(a, k=0, start=0, stop=None, key=None):
  quickfind_k(a, k, start, stop, True, key)
  return a

# http://en.wikipedia.org/wiki/Selection_algorithm#Optimised_sorting_algorithms
def quickfind_k(a, k=0, start=0, stop=None, sort=False, key=None):
  if stop is None:
    stop = len(a)
  if stop - start > 1:
    pivot = partition(a, start, stop, key)
    if k < pivot or sort:
      quickfind_k(a, k, start, pivot, sort, key)
    if pivot < k:
      quickfind_k(a, k, pivot + 1, stop, sort, key)
  return a

def partition(a, start, stop=None, key=None):
  if stop is None:
    stop = len(a)
  if key is None:
    key = lambda e: e
  pivot = start + random.randrange(stop - start)
  a[pivot], a[start], pivot = a[start], a[pivot], start
  for i in range(start + 1, stop):
    if key(a[i]) < key(a[start]):
      pivot += 1
      a[pivot], a[i] = a[i], a[pivot]
  a[pivot], a[start] = a[start], a[pivot]
  return pivot

for _ in range(1000):
  assert [0, 1] == quickfind_k([9, 8, 7, 6, 5, 4, 3, 2, 1, 0], 1)[:2]
  assert 2 == quickfind_k([9, 8, 7, 6, 5, 4, 3, 2, 1, 0], 2)[2]
  assert 3 == quickfind_k([9, 8, 7, 6, 5, 4, 3, 2, 1, 0], 3)[3]
  assert [0, 1, 2, 3] == quicksort_k([9, 8, 7, 6, 5, 4, 3, 2, 1, 0], 3)[:4]

In [ ]:
# @title ##### 17.15 Longest Word: Given a list of words, write a program to find the longest word made of other words in the list.
def longest_composition(L):
  vocas = dict.fromkeys(sorted(L, key=len, reverse=True))
  @lru_cache(maxsize=None)
  def prog(word, m):
    if m > 0:
      n = next((n for n in range(m - 1, 0, -1) if word[n:m] in vocas), None)
      if n:
        if word[:n] in vocas:
          yield (n, m)
        else:
          yield from (e + (m,) for e in prog(word, n))
    else:
      yield ()
  compose = lambda c: [word[i > 0 and c[i - 1] : stop] for i, stop in enumerate(c)]
  for word in vocas:
    yield from (compose(stops) for stops in prog(word, len(word)))

L =  ["cat", "cats", "catdogcats", "catcats", "dog", "dogcatsdog", "ratcatdogcats"]  # fmt: skip
assert ["cat", "dog", "cats"] == next(longest_composition(L), None)

In [ ]:
# @title ##### 17.16 A popular masseuse receives a sequence of back-to-back appointment requests and is debating which ones to accept. She needs a 15-minute break between appointments and therefore she cannot accept any adjacent requests. Given a sequence of back-to-back appointment requests (all multiples of 15 minutes, none overlap, and none can be moved), find the optimal (highest total booked minutes) set the masseuse can honor. Return the number of minutes.
def masseuse(L):  # L: massage appointmts in minutes.
  recap = lambda kv, k, v: (kv[0] + k, kv[1] + v)  # recap of (minutes, sequence).
  def prog(n):  # n appts. to process.
    if n > 0:
      key, value = L[n - 1], (n - 1,)  # key: minutes, value: seq. of decisions.
      cases = (recap(prog(n - 2), key, value), recap(prog(n - 1), 0, ()))
      return max(cases, key=lambda e: e[0])
    else:  # no appts. to process.
      return 0, ()
  return prog(len(L))

L = [30, 15, 60, 75, 45, 15, 15, 45]  # a list of appointmts.
minutes, indices = masseuse(L)
assert (180, (30, 60, 45, 45)) == (minutes, tuple(L[i] for i in indices))

In [ ]:
# @title ##### 17.17 Given a text T and a set of queries Q, write a program to search T for each query in Q.
# Related: the longest common substrings to a set of queries Q.
class Trie:  # Be Pythonic!
  def __init__(self, value=None):
    from collections import defaultdict
    self.value, self.children = value, defaultdict(lambda: Trie())
  def __setitem__(self, key, child):
    if len(key) > 1:
      self.children[key[0]][key[1:]] = child
    else:
      self.children[key[0]] = child if isinstance(child, Trie) else Trie(child)
  def __getitem__(self, key):
    return (c := self.children.get(key[0], None)) and c[key[1:]] if key else self
  def values(self):
    if self.value is not None:
      yield self.value
    for key, child in self.children.items():
      yield from child.values()

s, suffs = "bananas", Trie()  # a suffix tree is a trie of all the suffixes.
for i, _ in enumerate(s):
  suffs[s[i:]] = i
#
trie = Trie()
vocabs = ["brace", "brazil", "bread", "brew", "brag", "brain"]
for i, e in enumerate(vocabs):
  trie[e] = i  # or Trie(i).
#
L = "ba na nas s bas".split(" ")
expected = [[0], [2, 4], [4], [6], None]
assert expected == [(t := suffs[q]) and list(t.values()) for q in L]
assert (0, 1, 4, 5) == tuple(tree["bra"].values())
assert (2, 3) == tuple(tree["bre"].values())

In [ ]:
# @title ##### 17.18 Given two arrays, one shorter (with all distinct elements) and one longer. Find the shortest subarray in the longer array that contains all the elements in the shorter array.
# * https://leetcode.com/problems/minimum-window-substring/
# * https://leetcode.com/problems/sliding-window-maximum/
# * https://leetcode.com/problems/minimum-size-subarray-sum/
# * https://leetcode.com/problems/minimum-window-substring/
# * https://leetcode.com/problems/maximum-length-of-repeated-subarray/
def minima(iterable, key=None):
  iterator = iter(iterable)
  minima = [next(iterator, None)]
  key = key or (lambda e: e)
  min_key = minima and key(minima[0])
  for e in iterator:
    if (k := key(e)) < min_key:
      minima = [e]
      min_key = k
    elif k == min_key:
      minima.append(e)
  return minima

def minimum_window_containing_subset(set_, subset):
  def prog():
    from collections import Counter
    demand = Counter(subset)
    supply = Counter({k: 0 for k in demand})
    minima, min_key = None, None
    start = 0
    for stop, i in enumerate(L, 1):  # i: in, o: out of a window.
      if i not in supply:
        continue
      supply[i] += i in supply
      if supply >= demand:
        # Slide the window, while d is over-supplied or not in demand.
        while supply[o := L[start]] > demand.get(o, -1):
          supply[o] -= o in supply
          start += 1
        yield (start, stop)
  return minima(prog(), key=lambda ss: ss[1] - ss[0])

L = [7, 5, 9, 0, 2, 1, 3, 5, 7, 9, 1, 1, 5, 8, 8, 9, 7]
assert [(7, 11), (9, 13)] == minimum_window_containing_subset(L, [5, 1, 9])

17.19 Given an integer array from 1 to N appearing exactly once,
except for one number that is missing. How can you find the missing number in O(N) time and 0(1) space? What if there were two numbers missing?

17.20 Write a program that can quickly answer a median value, while random numbers are being generated and offered (a median bag).

In [ ]:
# @title ##### 17.21: Given n non-negative integers representing an elevation map where the width of each bar is 1, compute how much water it is able to trap after raining. For example, given [0,1,0,2,1,0,1,3,2,1,2,1], return 6.
def rain_water(elevations):
  from itertools import accumulate
  L = [0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]
  return sum(
    [
      min(*_) - height  # water in a column: min(prefix-max, suffix-max) -height.
      for (height, *_) in zip(
        L,
        accumulate(L, max),  # prefix-maxima
        reversed(list(accumulate(reversed(L), max))),  # suffix-maxima
      )
    ]
  )

assert 6 == rain_water([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1])

In [ ]:
# @title ##### 17.22 Given two words of equal length that are in a dictionary, write a method to transform one word into another word by changing only one letter at a time. The new word you get in each step must be in the dictionary.
def backtrack(candidate, expand_out, reduce_off):
  if not reduce_off(candidate):
    for e in expand_out(candidate):
      candidate.append(e)
      backtrack(candidate, expand_out, reduce_off)
      candidate.pop()

def transform(start, stop, dict_):
  def branch_out(a):
    word = list(a[-1])
    seen = set(a)
    for i, e in enumerate(word):
      import string
      for c in set(string.ascii_uppercase):
        word[i] = c
        y = "".join(word)
        if y not in seen and y in dict_:
          yield y
      word[i] = e
  def reduce_off(a):
    if a[-1] == stop:
      if reduce_off.reduced is None or len(reduce_off.reduced) > len(a):
        reduce_off.reduced = a[:]
      return True
  reduce_off.reduced = None
  backtrack([start], branch_out, reduce_off)
  return reduce_off.reduced

dict_ = set("CAMP;DAMP;LAMP;RAMP;LIMP;LUMP;LIMO;LITE;LIME;LIKE".split(";"))
assert ["DAMP", "LAMP", "LIMP", "LIME", "LIKE"] == transform("DAMP", "LIKE", dict_)

17.23 Given a square matrix of black and white cells, write a program to find the maximum sub-square such that all four borders are filled with black pixels.

In [ ]:
# @title ##### 17.24 Given a square matrix of positive and negatives integers, write a program to find the sub-matrix with the largest possible sum.

def maxsum_submatrix(g):
    m, n = len(g), len(g[0])
    prefix_sums_v = [[0] * n for _ in range(m)] # vertically
    for r in range(m):
        for c in range(n):
            prefix_sums_v[r][c] = (prefix_sums_v[r-1][c] if r > 0 else 0) + g[r][c]

    max_sum, max_matrix = g[0][0], (0, 0, 0, 0)
    for top in range(m):
        for bottom in range(top, m):
            sum_h = left = 0;
            for right in range(n):
                sum_v = (prefix_sums_v[bottom][right]
                         - (prefix_sums_v[top-1][right] if top > 0 else 0))
                if sum_h > 0:
                    sum_h += sum_v
                else:
                    sum_h, left = sum_v, right
                if sum_h >= max_sum:
                    max_sum, max_matrix = sum_h, (top, left, bottom, right)

    return (max_sum, max_matrix)

g = [ # 4 x 3 matrix
    [ 1,  0, 1],
    [ 0, -1, 0],
    [ 1,  0, 1],
    [-5,  2, 5]
]

assert (8, (2, 1, 3, 2)) == maxsum_submatrix(g)
assert (3, (0, 0, 0, 1)) == maxsum_submatrix([[1, 2, -1], [-3, -1, -4], [1, -5, 2]])

17.25 Given millions of words, write a program to create the largest possible rectangle of letters such that every row forms a word (reading left to right) and every column forms a word (reading top to bottom).



In [ ]:
# @title
def groupby(iterable, key=None, unique=False):
  d = {}
  if key is None:
    key = lambda e: e
  for e in iterable:
    k = key(e)
    if k in d:
      if unique:
        d[k].add(e)
      else:
        d[k].append(e)
    else:
      d[k] = set((e,)) if unique else [e]
  return d

def permutate(a, k=None, i=0):  # places values at indices from i to k-1.
  if k is None:
    k = len(a)
  if i == k:
    yield a[:k]
  else:
    for j in {a[j]: j for j in range(i, len(a))}.values():
      a[i], a[j] = a[j], a[i]
      yield from permutate(a, k, i + 1)
      a[i], a[j] = a[j], a[i]

assert {0: [2, 2], 1: [1, 3, 3, 3]} == groupby([1, 2, 2, 3, 3, 3], lambda e: e % 2)
assert {3: {"abc", "def"}, 4: {"wxyz"}} == groupby(["abc", "def", "wxyz"], len, True)
assert [[1, 1], [1, 2], [2, 1]] == sorted(permutate([1, 1, 2], 2))

def rectangle(words):
  d = groupby(words, len, unique=True)
  n = max(d.keys())
  area = n**2
  for a in range(area, 0, -1):
    for w in (w for w in range(n, 0, -1) if 0 == a % w):
      h = a // w  # height is area divided by width
      if w >= h and d.get(w) and len(d[w]) >= h:
        for p in permutate(list(d[w]), h):
          if all(
            ("".join((p[r][c] for r in range(h))) in d[h] for c in range(w))
          ):
            return p

words = set("a cat ate tea strafer taeniae resters antiflu fiefdom earlobe resumes schools coconut acacias oracle largest fingernails".split(" "))  # fmt: skip
assert 7 == len(rectangle(words))

17.26 Sparse Similarity: The similarity of two documents (each with distinct words) is defined to be the size of the intersection divided by the size of the union. For example, if the documents consist of integers, the similarity of {1, 5, 3} and {1, 7, 2, 3} is e. 4, because the intersection has size 2 and the union has size 5. We have a long list of documents (with distinct values and each with an associated ID) where the similarity is believed to be "sparse:'That is, any two arbitrarily selected documents are very likely to have similarity O. Design an algorithm that returns a list of pairs of document IDs and the associated similarity. Print only the pairs with similarity greater than O. Empty documents should not be printed at all. For simplicity, you may assume each document is represented as an array of distinct integers.